# Project 3 — Early Warning System for Sepsis (PhysioNet 2019)
## Notebook 02 — Leakage-safe Temporal Feature Engineering (Windowed Vitals)

**Goal**
Turn the long ICU time-series table into a model-ready feature matrix with **no label leakage**.

**Inputs (from Notebook 01)**
- `results/sample_long_table.parquet` — pre-onset rows only + `y_within_H`

**Outputs**
- `results/features_02.parquet` — engineered features + identifiers + target
- `results/feature_manifest.json` — feature list & configuration

**Leakage rule (non-negotiable)**
All features at time `t` must be computed using data from times **< t** only.
We enforce this using a **shift(1)** within each patient before applying rolling windows.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

# -------------------------
# Paths (EDIT THIS ONCE)
# -------------------------
PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT  # set this if your dataset/results live elsewhere

RESULTS_DIR = DATA_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IN_LONG = RESULTS_DIR / "sample_long_table.parquet"
OUT_FEATS = RESULTS_DIR / "features_02.parquet"
OUT_MANIFEST = RESULTS_DIR / "feature_manifest.json"

print("Input:", IN_LONG, "exists:", IN_LONG.exists())


Input: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\sample_long_table.parquet exists: True


### 1) Load long table
This should already be **pre-onset** rows only (Notebook 01 censored), and include:
- `patient_id`, `ICULOS`, `y_within_H`


In [2]:
df = pd.read_parquet(IN_LONG)
df = df.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

required = {"patient_id", "ICULOS", "y_within_H"}
missing_req = required - set(df.columns)
assert not missing_req, f"Missing required columns: {missing_req}"

# -------------------------
# HARD BAN: label/onset/helper columns (prevents leakage)
# -------------------------
BANNED_SUBSTRINGS = ["SepsisLabel", "event_iculos", "use_row"]  # anything containing these is not allowed as input
banned_cols = [c for c in df.columns if any(s in c for s in BANNED_SUBSTRINGS)]

if banned_cols:
    print("Dropping banned cols to prevent leakage:", banned_cols[:20], ("..." if len(banned_cols) > 20 else ""))
    df = df.drop(columns=banned_cols)

print("Rows:", len(df), "Patients:", df["patient_id"].nunique(), "Cols:", df.shape[1])
df.head()


Dropping banned cols to prevent leakage: ['SepsisLabel', 'event_iculos', 'use_row'] 
Rows: 19639 Patients: 494 Cols: 42


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,patient_id,y_within_H
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,1,p000214,0
1,76.0,100.0,NaN,143.0,91.0,65.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,2,p000214,0
2,76.0,100.0,37.33,155.0,99.0,71.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,3,p000214,0
3,74.0,100.0,NaN,140.0,91.0,66.0,14.5,NaN,NaN,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,4,p000214,0
4,76.0,100.0,NaN,146.0,97.0,69.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,5,p000214,0


### 2) Choose raw features to engineer
We exclude identifiers and label columns. Everything else becomes a candidate raw feature.


In [3]:
ID_COLS = ["patient_id", "ICULOS"]
LABEL_COLS = ["y_within_H"]

# Anything related to labels / onset / helper flags must never become a feature (or a __miss feature)
BANNED_SUBSTRINGS = ["SepsisLabel", "event_iculos", "use_row"]

def is_banned_feature(col: str) -> bool:
    if col in ID_COLS + LABEL_COLS:
        return False
    return any(s in col for s in BANNED_SUBSTRINGS)

raw_cols = [c for c in df.columns if (c not in ID_COLS + LABEL_COLS) and (not is_banned_feature(c))]

# sanity
assert not any(is_banned_feature(c) for c in raw_cols), "Banned columns still present in raw_cols."

print("Raw candidate columns:", len(raw_cols))
raw_cols[:20]


Raw candidate columns: 39


['HR',
 'O2Sat',
 'Temp',
 'SBP',
 'MAP',
 'DBP',
 'Resp',
 'EtCO2',
 'BaseExcess',
 'HCO3',
 'FiO2',
 'pH',
 'PaCO2',
 'SaO2',
 'AST',
 'BUN',
 'Alkalinephos',
 'Calcium',
 'Chloride',
 'Creatinine']

### 3) Optional: within-patient forward fill
ICU time-series are very sparse. Forward filling often helps, **but** we preserve missingness signals via flags.

- `ffill_within_patient=True`: forward fill each raw variable within each patient.
- Missingness flags are computed from the **original** missingness (before fill).


In [4]:
ffill_within_patient = True

df_raw = df[ID_COLS + LABEL_COLS + raw_cols].copy()

# Missingness flags before fill (strong predictors)
for c in raw_cols:
    df_raw[f"{c}__miss"] = df_raw[c].isna().astype(np.int8)

if ffill_within_patient:
    df_raw[raw_cols] = (
        df_raw.groupby("patient_id", sort=False)[raw_cols]
              .ffill()
    )

df_raw.head()


,patient_id,ICULOS,y_within_H,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,HR__miss,O2Sat__miss,Temp__miss,SBP__miss,MAP__miss,DBP__miss,Resp__miss,EtCO2__miss,BaseExcess__miss,HCO3__miss,FiO2__miss,pH__miss,PaCO2__miss,SaO2__miss,AST__miss,BUN__miss,Alkalinephos__miss,Calcium__miss,Chloride__miss,Creatinine__miss,Bilirubin_direct__miss,Glucose__miss,Lactate__miss,Magnesium__miss,Phosphate__miss,Potassium__miss,Bilirubin_total__miss,TroponinI__miss,Hct__miss,Hgb__miss,PTT__miss,WBC__miss,Fibrinogen__miss,Platelets__miss,Age__miss,Gender__miss,Unit1__miss,Unit2__miss,HospAdmTime__miss
0,p000214,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0
1,p000214,2,0,76.0,100.0,NaN,143.0,91.0,65.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,0,0,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0
2,p000214,3,0,76.0,100.0,37.33,155.0,99.0,71.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0
3,p000214,4,0,74.0,100.0,37.33,140.0,91.0,66.0,14.5,NaN,NaN,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,0,0,1,0,0,0,0,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0
4,p000214,5,0,76.0,100.0,37.33,146.0,97.0,69.0,15.0,NaN,NaN,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1,NaN,NaN,-13.07,0,0,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0


### 4) Leakage-safe temporal features (shift(1) + rolling windows)

We compute features using:
1) `shift(1)` inside each patient so current hour values don't leak into features
2) rolling windows on the shifted values:
   - mean, min, max, std over windows `[6, 12, 24]`
3) short-term change features:
   - `delta_1h` = x(t-1) - x(t-2)  (implemented on shifted series)
   - `delta_6h` = x(t-1) - x(t-7)

All of these use **past-only** information.


In [5]:
WINDOWS = [6, 12, 24]

def add_temporal_features(g, cols, windows):
    g = g.sort_values("ICULOS")
    shifted = g[cols].shift(1)  # prevent leakage: use t-1 and earlier only

    feats = {}

    # Rolling stats
    for w in windows:
        roll = shifted.rolling(window=w, min_periods=1)

        m = roll.mean().to_numpy()
        mn = roll.min().to_numpy()
        mx = roll.max().to_numpy()
        sd = roll.std(ddof=0).to_numpy()

        for j, c in enumerate(cols):
            feats[f"{c}__mean_{w}h"] = m[:, j]
            feats[f"{c}__min_{w}h"]  = mn[:, j]
            feats[f"{c}__max_{w}h"]  = mx[:, j]
            feats[f"{c}__std_{w}h"]  = sd[:, j]

    # Deltas (past-only)
    d1 = (shifted - shifted.shift(1)).to_numpy()
    d6 = (shifted - shifted.shift(6)).to_numpy()
    for j, c in enumerate(cols):
        feats[f"{c}__delta_1h"] = d1[:, j]
        feats[f"{c}__delta_6h"] = d6[:, j]

    # Last observed (t-1)
    prev1 = shifted.to_numpy()
    for j, c in enumerate(cols):
        feats[f"{c}__prev1"] = prev1[:, j]

    out = pd.DataFrame(feats, index=g.index)
    return out

# Only engineer numeric columns (most are numeric in this dataset)
num_cols = [c for c in raw_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
print("Numeric raw columns:", len(num_cols))

feat_blocks = []
for pid, g in df_raw.groupby("patient_id", sort=False):
    feats_g = add_temporal_features(g, num_cols, WINDOWS)
    feat_blocks.append(feats_g)

X_time = pd.concat(feat_blocks).sort_index()

print("Temporal feature matrix shape:", X_time.shape)
X_time.head()


Numeric raw columns: 39
Temporal feature matrix shape: (19639, 585)


,HR__mean_6h,HR__min_6h,HR__max_6h,HR__std_6h,O2Sat__mean_6h,O2Sat__min_6h,O2Sat__max_6h,O2Sat__std_6h,Temp__mean_6h,Temp__min_6h,Temp__max_6h,Temp__std_6h,SBP__mean_6h,SBP__min_6h,SBP__max_6h,SBP__std_6h,MAP__mean_6h,MAP__min_6h,MAP__max_6h,MAP__std_6h,DBP__mean_6h,DBP__min_6h,DBP__max_6h,DBP__std_6h,Resp__mean_6h,Resp__min_6h,Resp__max_6h,Resp__std_6h,EtCO2__mean_6h,EtCO2__min_6h,EtCO2__max_6h,EtCO2__std_6h,BaseExcess__mean_6h,BaseExcess__min_6h,BaseExcess__max_6h,BaseExcess__std_6h,HCO3__mean_6h,HCO3__min_6h,HCO3__max_6h,HCO3__std_6h,FiO2__mean_6h,FiO2__min_6h,FiO2__max_6h,FiO2__std_6h,pH__mean_6h,pH__min_6h,pH__max_6h,pH__std_6h,PaCO2__mean_6h,PaCO2__min_6h,PaCO2__max_6h,PaCO2__std_6h,SaO2__mean_6h,SaO2__min_6h,SaO2__max_6h,SaO2__std_6h,AST__mean_6h,AST__min_6h,AST__max_6h,AST__std_6h,BUN__mean_6h,BUN__min_6h,BUN__max_6h,BUN__std_6h,Alkalinephos__mean_6h,Alkalinephos__min_6h,Alkalinephos__max_6h,Alkalinephos__std_6h,Calcium__mean_6h,Calcium__min_6h,Calcium__max_6h,Calcium__std_6h,Chloride__mean_6h,Chloride__min_6h,Chloride__max_6h,Chloride__std_6h,Creatinine__mean_6h,Creatinine__min_6h,Creatinine__max_6h,Creatinine__std_6h,Bilirubin_direct__mean_6h,Bilirubin_direct__min_6h,Bilirubin_direct__max_6h,Bilirubin_direct__std_6h,Glucose__mean_6h,Glucose__min_6h,Glucose__max_6h,Glucose__std_6h,Lactate__mean_6h,Lactate__min_6h,Lactate__max_6h,Lactate__std_6h,Magnesium__mean_6h,Magnesium__min_6h,Magnesium__max_6h,Magnesium__std_6h,Phosphate__mean_6h,Phosphate__min_6h,Phosphate__max_6h,Phosphate__std_6h,...,BaseExcess__delta_6h,HCO3__delta_1h,HCO3__delta_6h,FiO2__delta_1h,FiO2__delta_6h,pH__delta_1h,pH__delta_6h,PaCO2__delta_1h,PaCO2__delta_6h,SaO2__delta_1h,SaO2__delta_6h,AST__delta_1h,AST__delta_6h,BUN__delta_1h,BUN__delta_6h,Alkalinephos__delta_1h,Alkalinephos__delta_6h,Calcium__delta_1h,Calcium__delta_6h,Chloride__delta_1h,Chloride__delta_6h,Creatinine__delta_1h,Creatinine__delta_6h,Bilirubin_direct__delta_1h,Bilirubin_direct__delta_6h,Glucose__delta_1h,Glucose__delta_6h,Lactate__delta_1h,Lactate__delta_6h,Magnesium__delta_1h,Magnesium__delta_6h,Phosphate__delta_1h,Phosphate__delta_6h,Potassium__delta_1h,Potassium__delta_6h,Bilirubin_total__delta_1h,Bilirubin_total__delta_6h,TroponinI__delta_1h,TroponinI__delta_6h,Hct__delta_1h,Hct__delta_6h,Hgb__delta_1h,Hgb__delta_6h,PTT__delta_1h,PTT__delta_6h,WBC__delta_1h,WBC__delta_6h,Fibrinogen__delta_1h,Fibrinogen__delta_6h,Platelets__delta_1h,Platelets__delta_6h,Age__delta_1h,Age__delta_6h,Gender__delta_1h,Gender__delta_6h,Unit1__delta_1h,Unit1__delta_6h,Unit2__delta_1h,Unit2__delta_6h,HospAdmTime__delta_1h,HospAdmTime__delta_6h,HR__prev1,O2Sat__prev1,Temp__prev1,SBP__prev1,MAP__prev1,DBP__prev1,Resp__prev1,EtCO2__prev1,BaseExcess__prev1,HCO3__prev1,FiO2__prev1,pH__prev1,PaCO2__prev1,SaO2__prev1,AST__prev1,BUN__prev1,Alkalinephos__prev1,Calcium__prev1,Chloride__prev1,Creatinine__prev1,Bilirubin_direct__prev1,Glucose__prev1,Lactate__prev1,Magnesium__prev1,Phosphate__prev1,Potassium__prev1,Bilirubin_total__prev1,TroponinI__prev1,Hct__prev1,Hgb__prev1,PTT__prev1,WBC__prev1,Fibrinogen__prev1,Platelets__prev1,Age__prev1,Gender__prev1,Unit1__prev1,Unit2__prev1,HospAdmTime__prev1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,Na

### 5) Combine identifiers + target + engineered features + missingness flags
We keep `patient_id` and `ICULOS` for grouped CV and for early-warning policy later.


In [6]:
# Collect miss flags
miss_cols = [c for c in df_raw.columns if c.endswith("__miss")]

out = pd.concat(
    [
        df_raw[ID_COLS + LABEL_COLS + miss_cols].reset_index(drop=True),
        X_time.reset_index(drop=True),
    ],
    axis=1
)

# Drop the first ICU hour per patient if desired (most features are NaN-ish there)
# We keep it, but you can drop it for modeling:
drop_first_hour = True
if drop_first_hour:
    first_mask = out.groupby("patient_id")["ICULOS"].transform("min") == out["ICULOS"]
    out = out.loc[~first_mask].reset_index(drop=True)

print("Final feature table:", out.shape)
out.head()


Final feature table: (19145, 627)


,patient_id,ICULOS,y_within_H,HR__miss,O2Sat__miss,Temp__miss,SBP__miss,MAP__miss,DBP__miss,Resp__miss,EtCO2__miss,BaseExcess__miss,HCO3__miss,FiO2__miss,pH__miss,PaCO2__miss,SaO2__miss,AST__miss,BUN__miss,Alkalinephos__miss,Calcium__miss,Chloride__miss,Creatinine__miss,Bilirubin_direct__miss,Glucose__miss,Lactate__miss,Magnesium__miss,Phosphate__miss,Potassium__miss,Bilirubin_total__miss,TroponinI__miss,Hct__miss,Hgb__miss,PTT__miss,WBC__miss,Fibrinogen__miss,Platelets__miss,Age__miss,Gender__miss,Unit1__miss,Unit2__miss,HospAdmTime__miss,HR__mean_6h,HR__min_6h,HR__max_6h,HR__std_6h,O2Sat__mean_6h,O2Sat__min_6h,O2Sat__max_6h,O2Sat__std_6h,Temp__mean_6h,Temp__min_6h,Temp__max_6h,Temp__std_6h,SBP__mean_6h,SBP__min_6h,SBP__max_6h,SBP__std_6h,MAP__mean_6h,MAP__min_6h,MAP__max_6h,MAP__std_6h,DBP__mean_6h,DBP__min_6h,DBP__max_6h,DBP__std_6h,Resp__mean_6h,Resp__min_6h,Resp__max_6h,Resp__std_6h,EtCO2__mean_6h,EtCO2__min_6h,EtCO2__max_6h,EtCO2__std_6h,BaseExcess__mean_6h,BaseExcess__min_6h,BaseExcess__max_6h,BaseExcess__std_6h,HCO3__mean_6h,HCO3__min_6h,HCO3__max_6h,HCO3__std_6h,FiO2__mean_6h,FiO2__min_6h,FiO2__max_6h,FiO2__std_6h,pH__mean_6h,pH__min_6h,pH__max_6h,pH__std_6h,PaCO2__mean_6h,PaCO2__min_6h,PaCO2__max_6h,PaCO2__std_6h,SaO2__mean_6h,SaO2__min_6h,SaO2__max_6h,SaO2__std_6h,AST__mean_6h,AST__min_6h,...,BaseExcess__delta_6h,HCO3__delta_1h,HCO3__delta_6h,FiO2__delta_1h,FiO2__delta_6h,pH__delta_1h,pH__delta_6h,PaCO2__delta_1h,PaCO2__delta_6h,SaO2__delta_1h,SaO2__delta_6h,AST__delta_1h,AST__delta_6h,BUN__delta_1h,BUN__delta_6h,Alkalinephos__delta_1h,Alkalinephos__delta_6h,Calcium__delta_1h,Calcium__delta_6h,Chloride__delta_1h,Chloride__delta_6h,Creatinine__delta_1h,Creatinine__delta_6h,Bilirubin_direct__delta_1h,Bilirubin_direct__delta_6h,Glucose__delta_1h,Glucose__delta_6h,Lactate__delta_1h,Lactate__delta_6h,Magnesium__delta_1h,Magnesium__delta_6h,Phosphate__delta_1h,Phosphate__delta_6h,Potassium__delta_1h,Potassium__delta_6h,Bilirubin_total__delta_1h,Bilirubin_total__delta_6h,TroponinI__delta_1h,TroponinI__delta_6h,Hct__delta_1h,Hct__delta_6h,Hgb__delta_1h,Hgb__delta_6h,PTT__delta_1h,PTT__delta_6h,WBC__delta_1h,WBC__delta_6h,Fibrinogen__delta_1h,Fibrinogen__delta_6h,Platelets__delta_1h,Platelets__delta_6h,Age__delta_1h,Age__delta_6h,Gender__delta_1h,Gender__delta_6h,Unit1__delta_1h,Unit1__delta_6h,Unit2__delta_1h,Unit2__delta_6h,HospAdmTime__delta_1h,HospAdmTime__delta_6h,HR__prev1,O2Sat__prev1,Temp__prev1,SBP__prev1,MAP__prev1,DBP__prev1,Resp__prev1,EtCO2__prev1,BaseExcess__prev1,HCO3__prev1,FiO2__prev1,pH__prev1,PaCO2__prev1,SaO2__prev1,AST__prev1,BUN__prev1,Alkalinephos__prev1,Calcium__prev1,Chloride__prev1,Creatinine__prev1,Bilirubin_direct__prev1,Glucose__prev1,Lactate__prev1,Magnesium__prev1,Phosphate__prev1,Potassium__prev1,Bilirubin_total__prev1,TroponinI__prev1,Hct__prev1,Hgb__prev1,PTT__prev1,WBC__prev1,Fibrinogen__prev1,Platelets__prev1,Age__prev1,Gender__prev1,Unit1__prev1,Unit2__prev1,HospAdmTime__prev1
0,p000214,2,0,0,0,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1.0,NaN,NaN,-13.07
1,p000214,3,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0,76.000000,76.0,76.0,0.000000,100.0,100.0,100.0,0.0,NaN,NaN,NaN,NaN,143.0,143.0,143.0,0.000000,91.000000,91.0,91.0,0.000000,65.000000,65.0,65.0,0.000000,16.000,16.0,16.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

### 6) Leakage sanity checks
We ensure:
- No raw (current-time) columns are accidentally included
- Feature columns follow expected naming
- No patient_id mixing


In [7]:
# Ensure raw cols are NOT present (we only want engineered + miss flags)
present_raw = [c for c in raw_cols if c in out.columns]
assert len(present_raw) == 0, f"Raw columns leaked into output: {present_raw[:10]}"

# Ensure label + IDs present
for c in ID_COLS + LABEL_COLS:
    assert c in out.columns, f"Missing column: {c}"

# Quick check: no duplicate rows for same (patient_id, ICULOS)
dups = out.duplicated(subset=["patient_id", "ICULOS"]).sum()
assert dups == 0, f"Duplicate (patient_id, ICULOS) rows: {dups}"

# HARD check: no banned substrings anywhere in output columns
BANNED_SUBSTRINGS = ["SepsisLabel", "event_iculos", "use_row"]
bad_out = [c for c in out.columns if any(s in c for s in BANNED_SUBSTRINGS)]
assert len(bad_out) == 0, f"Leakage columns present in output: {bad_out[:20]}"

print("Leakage checks passed ✅")


Leakage checks passed ✅


### 7) Save artifacts
We save parquet + a manifest describing:
- windows used
- ffill setting
- feature column names


In [8]:
out.to_parquet(OUT_FEATS, index=False)

feature_cols = [c for c in out.columns if c not in ID_COLS + LABEL_COLS]
manifest = {
    "created_from": str(IN_LONG),
    "windows": WINDOWS,
    "ffill_within_patient": bool(ffill_within_patient),
    "drop_first_hour": bool(drop_first_hour),
    "n_rows": int(out.shape[0]),
    "n_patients": int(out["patient_id"].nunique()),
    "n_features": int(len(feature_cols)),
    "id_cols": ID_COLS,
    "label_cols": LABEL_COLS,
    "feature_cols_preview": feature_cols[:50],
}

OUT_MANIFEST.write_text(json.dumps(manifest, indent=2))
print("Saved:")
print(" -", OUT_FEATS)
print(" -", OUT_MANIFEST)


Saved:
 - C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\features_02.parquet
 - C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\feature_manifest.json
